# DL_classification_image

8×8 필기체 이미지로 숫자 0~9를 분류합니다.


## 1. 라이브러리와 재현성
고정 시드를 사용하지만 실행 장치와 라이브러리 버전에 따라 결과가 달라질 수 있습니다.


In [ ]:
import os, sys, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay,
    mean_absolute_error, mean_squared_error, r2_score)

SEED = 42
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
EPOCHS = 50
BATCH_SIZE = 32
print('Python:', sys.version.split()[0], 'TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 2. 데이터 불러오기


In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
original_images = digits.images.astype('float32')
print('Image shape:', original_images.shape, 'pixel range:', original_images.min(), original_images.max())
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(original_images[i], cmap='gray'); ax.set_title(str(digits.target[i])); ax.axis('off')
plt.tight_layout(); plt.show()


## 3. 입력 X와 정답 y


In [ ]:
y = digits.target.astype('int32')
class_names = np.arange(10)
STRATIFY = True


## 4. 데이터 분할 및 전처리
전체의 약 64%/16%/20%를 훈련/검증/테스트로 사용합니다. 테스트는 최종 평가에만 사용합니다.


In [ ]:
# 먼저 분할한 뒤, 훈련 데이터에서만 대치값/표준화 기준을 학습합니다.
idx = np.arange(len(y))
trainval_idx, test_idx = train_test_split(idx, test_size=0.2,
    random_state=SEED, stratify=y if STRATIFY else None)
train_idx, val_idx = train_test_split(trainval_idx, test_size=0.2,
    random_state=SEED, stratify=y[trainval_idx] if STRATIFY else None)
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
print('Train / validation / test:', len(train_idx), len(val_idx), len(test_idx))
assert not set(train_idx) & set(test_idx)
assert not set(val_idx) & set(test_idx)


In [ ]:
# load_digits 픽셀값은 0~16입니다. MNIST의 0~255와 다릅니다.
X_train = (original_images[train_idx] / 16.0).astype('float32')[..., None]
X_val = (original_images[val_idx] / 16.0).astype('float32')[..., None]
X_test = (original_images[test_idx] / 16.0).astype('float32')[..., None]


## 5. 분류 모델
이진분류는 Sigmoid + Binary Crossentropy, 다중분류는 Softmax + Sparse Categorical Crossentropy를 사용합니다. Sparse 손실은 원-핫 대신 0부터 시작하는 정수 클래스 인덱스를 받습니다.


In [ ]:
y_train_fit, y_val_fit = y_train, y_val
METRIC = 'accuracy'
unseen = sorted(set(y_test) - set(y_train))
print('훈련에 없는 테스트 클래스:', [class_names[i] for i in unseen])
print('희소 클래스는 성능 추정이 불안정합니다. 보고서의 support도 확인하세요.')


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=X_train.shape[1:]),
    tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy', metrics=[METRIC])
model.summary()


## 6. 학습
`fit()`에서 순전파, 손실 계산, 역전파, 가중치 갱신을 반복합니다. 검증 데이터로 조기 종료 시점을 선택합니다.


In [ ]:
# 검증 손실이 개선되지 않으면 중단하고 최적 가중치를 복원합니다.
callbacks = [tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True)]
history = model.fit(X_train, y_train_fit,
    validation_data=(X_val, y_val_fit), epochs=EPOCHS,
    batch_size=BATCH_SIZE, callbacks=callbacks, verbose=2)


## 7. 학습 곡선
훈련 손실은 감소하지만 검증 손실이 증가하면 과적합을 의심합니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for key, ax in zip(['loss', METRIC], axes):
    ax.plot(history.history[key], label='Train')
    ax.plot(history.history['val_' + key], label='Validation')
    ax.set(xlabel='Epoch (zero-based)', ylabel=key, title=key)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 8. 독립 테스트 평가


In [ ]:
probabilities = model.predict(X_test, verbose=0)
y_pred = probabilities.argmax(axis=1)
metrics = {'accuracy': float(accuracy_score(y_test, y_pred)),
           'macro_f1': float(f1_score(y_test, y_pred, labels=np.arange(len(class_names)), average='macro', zero_division=0))}
majority = np.bincount(y_train).argmax()
metrics['majority_baseline_accuracy'] = float(np.mean(y_test == majority))
print(metrics)
print(classification_report(y_test, y_pred, labels=np.arange(len(class_names)),
      target_names=[str(v) for v in class_names], zero_division=0))
fig, ax = plt.subplots(figsize=(12, 10) if len(class_names)>10 else (7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    labels=np.arange(len(class_names)), display_labels=class_names, ax=ax,
    cmap='Blues', include_values=len(class_names)<=10, colorbar=False)
ax.set_title('Test confusion matrix'); plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_test[i, ..., 0], cmap='gray')
    ax.set_title(f'True: {y_test[i]:.1f} / Pred: {y_pred[i]:.1f}')
    ax.axis('off')
plt.tight_layout(); plt.show()


## 9. 결과 저장
Colab 파일 탭에서 모델·지표·예측 CSV를 다운로드할 수 있습니다. 모델 재사용 시 전처리 파일도 함께 보관하세요.


In [ ]:
import joblib
RESULT_DIR = Path('DL_classification_image_results')
RESULT_DIR.mkdir(exist_ok=True)
model.save(RESULT_DIR / 'model.keras')
(RESULT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
pd.DataFrame({'actual': y_test, 'predicted': y_pred}).to_csv(RESULT_DIR / 'predictions.csv', index=False)
metadata = {'task': 'classification', 'seed': SEED, 'tensorflow': tf.__version__}
if 'preprocessor' in globals(): joblib.dump(preprocessor, RESULT_DIR / 'preprocessor.joblib')
if 'target_scaler' in globals(): joblib.dump(target_scaler, RESULT_DIR / 'target_scaler.joblib')
if 'class_names' in globals(): metadata['class_names'] = [str(x) for x in class_names]
(RESULT_DIR / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', RESULT_DIR)


## 참고 자료
- [사용자 3주차 원본](https://github.com/namgiju/mlprog/blob/main/3%EC%A3%BC%EC%B0%A8/chapter03_ml_organized.ipynb)
- [scikit-learn digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)
